In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np #numpy library added 

# Small text dataset (context)
context = [
    "The sky is blue and the sun is bright.",
    "Cats are independent animals, but they love attention."
]

train_data = [
    ("What color do you think of the sky?", "blue"),
    ("What do cats love or like?", "attention")
]

# Tokenizer (simplistic)
def tokenize(text):
    return text.lower().split()

# Vocabulary and mappings
vocab = list(set(token for sentence in context + [q for q, a in train_data] + [a for q, a in train_data] for token in tokenize(sentence)))
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

# Encode sentences
def encode(sentence):
    tokens = tokenize(sentence)
    return torch.tensor([word_to_idx[token] for token in tokens], dtype=torch.long)

# Define a simple attention-based model
import torch
import torch.nn as nn
import numpy as np

class AttentionQA(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.output_proj = nn.Linear(embed_dim, vocab_size)

    def forward(self, context, question):
        # Embedding
        context_embed = self.embedding(context)  
        question_embed = self.embedding(question)  

        #  the numpy matrix based implementation
        C = context_embed.detach().cpu().numpy()   
        Q = question_embed.detach().cpu().numpy()  

        W_Q = self.query_proj.weight.detach().cpu().numpy().T
        b_Q = self.query_proj.bias.detach().cpu().numpy()
        W_K = self.key_proj.weight.detach().cpu().numpy().T
        b_K = self.key_proj.bias.detach().cpu().numpy()
        W_V = self.value_proj.weight.detach().cpu().numpy().T
        b_V = self.value_proj.bias.detach().cpu().numpy()

        Q_mat = Q @ W_Q + b_Q         
        K_mat = C @ W_K + b_K         
        V_mat = C @ W_V + b_V         

        scores = Q_mat @ K_mat.T / np.sqrt(K_mat.shape[1])  
        scores -= scores.max(axis=1, keepdims=True)
        weights = np.exp(scores)
        weights /= weights.sum(axis=1, keepdims=True)       

        attended_np = weights @ V_mat                        
        attended = torch.tensor(attended_np, dtype=torch.float32, device=context_embed.device)

        context_vector = attended.mean(axis=0)             

        # Output projection
        logits = self.output_proj(context_vector)  
        return logits


# Hyperparameters
embed_dim = 16
model = AttentionQA(len(vocab), embed_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(100):
    total_loss = 0
    for question, answer in train_data:
        context_tensor = encode(" ".join(context))
        question_tensor = encode(question)
        answer_tensor = torch.tensor(word_to_idx[answer], dtype=torch.long)

        logits = model(context_tensor, question_tensor)
        loss = criterion(logits, answer_tensor)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

# Testing
test_questions = [
    "What is the color of the sky?",
    "What do cats like?"
]

correct_answers = ["blue", "attention"]

print("\nTesting:")
for question, correct_answer in zip(test_questions, correct_answers):
    question_tensor = encode(question)
    context_tensor = encode(" ".join(context))
    logits = model(context_tensor, question_tensor)
    predicted_idx = torch.argmax(logits).item()
    predicted_answer = idx_to_word[predicted_idx]

    print(f"Question: {question}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Correct: {predicted_answer == correct_answer}\n")

Epoch 20, Loss: 3.7769
Epoch 40, Loss: 2.2416
Epoch 60, Loss: 1.7591
Epoch 80, Loss: 1.5664
Epoch 100, Loss: 1.4594

Testing:
Question: What is the color of the sky?
Predicted Answer: blue
Correct Answer: blue
Correct: True

Question: What do cats like?
Predicted Answer: attention
Correct Answer: attention
Correct: True

